# Module 3: Interference and the Fixed-Rate Bracket

A manager can ask for an intermediate assessment before a decision or request the decision directly. These cells distinguish the assessed accounting identity from an assumption that carries its conditional rates into the direct procedure.

## Probability Functions

Run this cell once. The complete helpers are included so the notebook runs by itself. Angles use degrees; table probabilities use the 0–1 scale.

In [1]:
"""Small quantum-probability examples for Modules 1–4.

Adapted from Nathan Yang's quantum cognition manuscript, ch_python_lab.tex.
All numerical examples are imposed models. They are not observations of people.
Angles passed to question() and state() are in degrees.
"""
import numpy as np

def state(angle_degrees):
    """A normalized real two-dimensional state at a stated angle."""
    theta=np.deg2rad(float(angle_degrees))
    if not np.isfinite(theta): raise ValueError('Angle must be finite.')
    return np.array([np.cos(theta),np.sin(theta)])

def _unit(psi):
    psi=np.asarray(psi,dtype=complex)
    if psi.ndim!=1 or not np.all(np.isfinite(psi)):
        raise ValueError('State must be a finite one-dimensional vector.')
    if not np.isclose(np.vdot(psi,psi).real,1,atol=1e-10):
        raise ValueError('State must have squared length one.')
    return psi

def projector(vector):
    """Projector onto a nonzero vector, normalized inside this function."""
    v=np.asarray(vector,dtype=complex)
    if v.ndim!=1 or not np.all(np.isfinite(v)) or np.linalg.norm(v)<1e-12:
        raise ValueError('Supply a finite nonzero vector.')
    v=v/np.linalg.norm(v)
    return np.outer(v,v.conj())

def question(angle_degrees):
    """Return yes and no projectors. Outcome index 0 means yes; 1 means no."""
    yes=projector(state(angle_degrees))
    return yes,np.eye(2)-yes

def born(psi,P):
    """Squared projected length for an orthogonal projector P."""
    psi=_unit(psi);P=np.asarray(P,dtype=complex)
    if P.shape!=(len(psi),len(psi)) or not np.all(np.isfinite(P)):
        raise ValueError('Projector dimensions must match the state.')
    if not np.allclose(P,P.conj().T,atol=1e-10) or not np.allclose(P@P,P,atol=1e-10):
        raise ValueError('P must be an orthogonal projector.')
    projected=P@psi
    return float(np.clip(np.vdot(projected,projected).real,0,1))

def update(psi,P):
    """State conditional on a possible observed outcome.

    A zero-probability outcome has no conditional state. Raise an error rather
    than divide by zero. joint() handles impossible branches without updating.
    """
    p=born(psi,P)
    if p<1e-14: raise ValueError('Cannot condition on a zero-probability outcome.')
    return P@np.asarray(psi,dtype=complex)/np.sqrt(p)

def joint(psi,first,second):
    """Rows are first-question yes/no, columns are second-question yes/no."""
    out=np.zeros((2,2))
    for i,P in enumerate(first):
        p=born(psi,P)
        if p<1e-14: continue
        posterior=update(psi,P)
        for j,Q in enumerate(second): out[i,j]=p*born(posterior,Q)
    if not np.isclose(out.sum(),1): raise ValueError('Questions must partition the state space.')
    return out

def qq(ab,ba):
    """Agreement-share difference. Input tables must each sum to one.

    AB: A rows, B columns. BA: B rows, A columns. Transposing BA to put
    A in rows leaves its diagonal sum unchanged.
    """
    tables=[np.asarray(t,dtype=float) for t in (ab,ba)]
    for t in tables:
        if t.shape!=(2,2) or not np.all(np.isfinite(t)) or (t<0).any() or not np.isclose(t.sum(),1):
            raise ValueError('Use a nonnegative 2 by 2 probability table summing to one.')
    return float(np.trace(tables[0])-np.trace(tables[1]))

def assessed_rate(psi,assessment,action):
    """Marginal yes rate for action after recording the assessment outcome."""
    return float(joint(psi,assessment,action)[:,0].sum())

def mixture(weight,rate_yes,rate_no):
    """Weighted rate under fixed conditional rates; all arguments are probabilities."""
    values=np.asarray([weight,rate_yes,rate_no],dtype=float)
    if not np.isfinite(values).all() or ((values<0)|(values>1)).any():
        raise ValueError('Inputs must lie in [0,1].')
    return float(weight*rate_yes+(1-weight)*rate_no)

def classical_order_example():
    """An explicit classical sequential process with an order effect and q=0.

    A first: P(A=yes)=.4; P(B=yes|A=yes)=.75; P(B=yes|A=no)=1/3.
    B first: P(B=yes)=.5; P(A=yes|B=yes)=.9; P(A=yes|B=no)=.5.
    These are separate context-dependent transition rules, not one unchanged
    joint distribution carried across the two procedures.
    """
    ab=np.array([[.4*.75,.4*.25],[.6/3,.6*2/3]])
    ba=np.array([[.5*.9,.5*.1],[.5*.5,.5*.5]])
    return ab,ba

def examples():
    psi=state(20);A=question(0);B=question(50)
    ab,ba=joint(psi,A,B),joint(psi,B,A)
    classical_ab,classical_ba=classical_order_example()
    coherent=state(45);assessment=question(0)
    action=question(np.rad2deg(np.arccos(np.sqrt(.7))))
    direct=born(coherent,action[0]);assessed=assessed_rate(coherent,assessment,action)
    return {
        'probability_at_37_degrees':born(state(37),question(0)[0]),
        'geometric_B_first':born(psi,B[0]),
        'geometric_B_after_A':float(ab[:,0].sum()),
        'geometric_q':qq(ab,ba),
        'classical_A_first':float(classical_ab[0,:].sum()),
        'classical_A_after_B':float(classical_ba[:,0].sum()),
        'classical_q':qq(classical_ab,classical_ba),
        'direct_rate':direct,'assessed_rate':assessed,
        'direct_minus_assessed':direct-assessed,
        'conditional_rates':[.7,.3],
    }



## Every Mixture of Two Fixed Rates

The conditional action rates are 0.70 and 0.30. Vary the weight while retaining those rates.

In [2]:
for w in [0,.25,.5,.75,1]:
    print(f'Weight {w:.2f}: assessed rate {mixture(w,.7,.3):.3f}')

Weight 0.00: assessed rate 0.300
Weight 0.25: assessed rate 0.400
Weight 0.50: assessed rate 0.500
Weight 0.75: assessed rate 0.600
Weight 1.00: assessed rate 0.700


## A Geometric Direct Procedure

A balanced state and a specified action direction produce a direct rate above both conditional rates. These are imposed mathematical inputs.

In [3]:
psi=state(45)
A=question(0)
F=question(np.rad2deg(np.arccos(np.sqrt(.7))))
r_yes=born(update(psi,A[0]),F[0])
r_no=born(update(psi,A[1]),F[0])
w=born(psi,A[0])
weighted=mixture(w,r_yes,r_no)
direct=born(psi,F[0])
print(f'w={w:.3f}; r_yes={r_yes:.3f}; r_no={r_no:.3f}')
print(f'Weighted={weighted:.6f}; direct={direct:.6f}; difference={direct-weighted:.6f}')

w=0.500; r_yes=0.700; r_no=0.300
Weighted=0.500000; direct=0.958258; difference=0.458258


## The Direction Can Reverse

Reflect the action direction across the horizontal axis. The assessed conditional rates remain the same, while the direct probability changes.

In [4]:
F_reflected=question(-np.rad2deg(np.arccos(np.sqrt(.7))))
print('Reflected direct:',born(psi,F_reflected[0]))
print('Reflected assessed:',assessed_rate(psi,A,F_reflected))

Reflected direct: 0.041742430504416055
Reflected assessed: 0.5


## Compare Explanations

A classical model can assign a separate response rate to the direct procedure. It then abandons the assumption that the assessed conditional rates carry over unchanged. State what additional restrictions would make that alternative testable.

The intermediate judgment may also affect time, accuracy, and reliance on advice. A higher action rate alone does not determine which design is better. Run M3A2 to examine uncertainty with simulated records. Source: Nathan Yang, Python lab and Meridian case.